# Nuisance parameters: plain D against profiled $D_s$

This notebook accompanies the docs page
[`nuisance-profiled-ds`](../../docs/examples/nuisance-profiled-ds.md). A signal fraction is
measured while two background shapes float, and the question is what changes when the binning
criterion is told which parameter will actually be published. The docs page tells the story at a
small sample size; this notebook runs the full study, prints every table, and re-renders the
committed figure.

Set `SCOREQUANT_EXAMPLE_FAST=1` to shrink every sample and optimizer budget for a quick pass.

## The problem

The reference density on the unit interval is a mixture of one narrow signal peak and two
truncated-exponential backgrounds with different rates. Every component is an exact normalized
density and the coefficients sum to one, so the first coefficient is literally the signal
fraction. Each event's score is the exact linear component score, so score column 0 is the
interest column and columns 1 and 2 are the nuisance columns — the layout `ProfiledDOptimality`
expects.

Double precision is an application-level choice. The library never sets it at import time, so the
notebook turns it on itself, before anything computes.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import scorequant as sq
from examples.nuisance_profiled_ds import (
    HEADLINE_BINS,
    build_problem,
    make_figure,
    run_study,
    score_labeling,
    unbinned_profiled_information,
)

problem = build_problem(n_bins=HEADLINE_BINS)
train, test = problem.train, problem.test
n_bins = problem.n_bins

{
    "training events": int(train.scores.shape[0]),
    "held-out events": int(test.scores.shape[0]),
    "bin budget": n_bins,
    "components": problem.component_names,
    "interest columns": problem.interest,
    "nuisance columns": problem.nuisance,
    "reference signal fraction": float(problem.coefficients[0]),
}

## The same sample under two criteria

One call each; the only difference is the criterion object. Both labelings are then scored twice,
by the same two public functions: `information_report` for the whole three-parameter Fisher
matrix, and `profiled_information_report` for the information about the fraction alone, after the
two background columns have been Schur-completed out.

In [ ]:
config = sq.DExchangeConfig(seed=11)

plain = sq.optimize_partition(
    train.scores,
    weights=train.weights,
    n_bins=n_bins,
    criterion=sq.DOptimality(),
    config=config,
)
profiled = sq.optimize_partition(
    train.scores,
    weights=train.weights,
    n_bins=n_bins,
    criterion=sq.ProfiledDOptimality(problem.interest),
    config=config,
)

header = f"{'labeling':<34}{'all three':>12}{'the fraction':>15}"
print(header)
print("-" * len(header))
for name, result in (("plain D", plain), ("profiled D_s", profiled)):
    scored = score_labeling(
        train.scores,
        np.asarray(result.labels),
        train.weights,
        interest=problem.interest,
        n_bins=n_bins,
    )
    print(f"{name:<34}{scored.full_retention:>12.5f}{scored.profiled_retention:>15.5f}")

Each criterion wins on its own objective and loses on the other. Profiled $D_s$ is not a better
criterion; it answers a different question and pays for the answer with information about
parameters nobody will quote.

## The certified ceiling, and its labels as an initializer

Neither number above says how close either partition is to the best rule that exists at this
budget. `efficient_score_bound` answers that with a certificate: it builds the full-data
efficient score, partitions that one-dimensional coordinate exactly by weighted interval dynamic
programming, and returns a ceiling on the profiled objective of every rule of the whole score
space with at most this many cells.

The labels attaining the ceiling already solve the relaxed problem, so they are also the natural
starting point for profiled exchange.

In [ ]:
bound = sq.efficient_score_bound(
    train.scores, interest=problem.interest, weights=train.weights, n_bins=n_bins
)
initialized = sq.optimize_partition(
    train.scores,
    weights=train.weights,
    n_bins=n_bins,
    criterion=sq.ProfiledDOptimality(problem.interest),
    config=config,
    initial_labels=bound.labels,
)
unbinned = unbinned_profiled_information(train.scores, train.weights, interest=problem.interest)

print(f"certified ceiling            {bound.upper_bound:.6f}")
print(f"ceiling as retention         {float(np.exp(bound.upper_bound - np.log(unbinned))):.6f}")
print(
    f"generic seeding              gap {bound.gap_to(profiled):.6f}   "
    f"scans {profiled.scans:3d}   relocations {profiled.accepted_moves:5d}"
)
print(
    f"started from the ceiling     gap {bound.gap_to(initialized):.6f}   "
    f"scans {initialized.scans:3d}   relocations {initialized.accepted_moves:5d}"
)

A profiled partition cannot be compiled into a reusable rule, and the library says so rather than
inventing one. That refusal is the whole content of the missing bridge: an exchange-stable D
partition is guaranteed to be the training realization of a nearest-cell rule, and a profiled one
is not.

In [ ]:
try:
    initialized.compile_quantizer()
except ValueError as error:
    print(error)

## The whole study

`run_study` reruns everything above at the full sample size, sweeps the bin budget against the
ceiling, fits a reusable rule under each criterion, and fits the signal fraction from binned
counts. It is the same function that regenerates the committed JSON and figure.

In [ ]:
study = run_study()
metrics = study.metrics

header = f"{'labeling':<38}{'all three':>12}{'the fraction':>15}"
print(header)
print("-" * len(header))
for row in metrics["partitions"]:
    print(f"{row['label']:<38}{row['full_retention']:>12.5f}{row['profiled_retention']:>15.5f}")
print()
header = f"{'reusable rule':<38}{'train, fraction':>17}{'held out, fraction':>21}"
print(header)
print("-" * len(header))
for row in metrics["rules"]:
    print(
        f"{row['label']:<38}{row['train_profiled_retention']:>17.5f}"
        f"{row['test_profiled_retention']:>21.5f}"
    )

The two criteria need different solvers for the held-out column, and the reason is a theorem
rather than a preference. The plain-D rule is the compiled exchange partition; the profiled rule
is a soft Voronoi fit, because finite profiled exchange has no canonical extension. The soft rule
is a restricted family, so it does not reach the free-label profiled partition — that gap is the
honest price of insisting on a rule.

## The ceiling across bin budgets

In [ ]:
header = (
    f"{'bins':>5}{'plain D':>10}{'D_s seeded':>13}{'D_s from ceiling':>18}"
    f"{'ceiling':>10}{'relocations':>13}"
)
print(header)
print("-" * len(header))
for row in metrics["ceiling_sweep"]:
    relocations = f"{int(row['seeded_moves'])} -> {int(row['initialized_moves'])}"
    print(
        f"{int(row['n_bins']):>5}{row['d_profiled_retention']:>10.5f}"
        f"{row['ds_seeded_retention']:>13.5f}{row['ds_initialized_retention']:>18.5f}"
        f"{row['ceiling_retention']:>10.5f}{relocations:>13}"
    )

Two different things happen along that sweep, and they are worth keeping apart. At the smallest
budgets the two profiled runs converge on the same labeling and the initializer only makes it
cheaper. At larger budgets the initializer also lands somewhere strictly better. Neither effect
is guaranteed; the certificate is what turns "it converged" into a statement about how much is
left on the table.

## What the measurement finally reports

Binning is only worth arguing about if it changes an answer. The fit below is the extended
Poisson likelihood over bin counts whose Fisher information is exactly the matrix the library has
been optimizing: the signal fraction is scanned and both background coefficients are profiled out
at every scan point. The data are the expected counts at the reference coefficients, so the
interval is the one the binning implies asymptotically rather than the outcome of one simulated
experiment.

In [ ]:
reference = float(problem.coefficients[0])
unbinned_width = float(metrics["intervals"]["unbinned_half_width"])
print(f"{'binning':<22}{'half-width':>13}{'from Fisher':>14}{'excess':>10}")
print("-" * 59)
print(f"{'unbinned reference':<22}{unbinned_width:>13.6f}{unbinned_width:>14.6f}{'-':>10}")
for row in metrics["intervals"]["rows"]:
    excess = row["half_width"] / unbinned_width - 1.0
    print(
        f"{row['label']:<22}{row['half_width']:>13.6f}"
        f"{row['fisher_half_width']:>14.6f}{100 * excess:>9.2f}%"
    )

fig, ax = plt.subplots(figsize=(6.5, 4))
for row in study.intervals.rows:
    ax.plot(study.intervals.values, study.intervals.curves[row.key], label=row.label)
ax.axhline(1.0, color="grey", linestyle=":", linewidth=0.9)
ax.axvline(reference, color="grey", linestyle=":", linewidth=0.9)
ax.set(
    ylim=(0, 4),
    xlabel="signal fraction",
    ylabel="-2 log likelihood ratio",
    title="Binned profile-likelihood scan",
)
ax.legend();

The scanned half-width reproduces the reciprocal square root of the binned profiled information
to several digits, which is what makes this fit a check on the library rather than an
illustration beside it.

## The committed figure

In [ ]:
figure = make_figure(study)
figure

## Interpretation

Three things separate cleanly on this problem.

Telling the criterion which parameter matters changes the partition, not just its bookkeeping:
the two labelings place rows together differently, and each wins decisively on its own objective
while losing on the other. Profiling buys information about the fraction by spending information
about parameters that will never be quoted, which is the right trade exactly when that is true.

The certified ceiling turns a local search into a bounded one. It is cheap, it is a genuine
ceiling for every rule at the budget rather than an estimate, and its labels are a strong
initializer — worth up to two orders of magnitude fewer relocations, and sometimes a strictly
better optimum.

And the downstream effect is real but modest: at four cells the profiled binning removes about
three fifths of the precision that binning costs, and at a generous bin budget it would remove
much less because there would be much less to remove. The reason to use it is that it costs
nothing, and the certificate tells you when to stop.

There is no compile bridge for profiled $D_s$, and there will not be one. A profiled partition is
a fact about the rows in hand; a profiled rule is a different object that has to be fitted as
one, which is what the soft Voronoi fit above does.